#### I haven't been able to create a model that accureately predicts Fantasy Score so now my plan is to try and create a smaller model for each variable that factors into the Yahoo Fantasy Points.

# Hitter Model

### Checking for Correlations with new_win

In [2]:
import pandas as pd
# Creating initial Data Frame
df = pd.read_csv("Hitter_Stats.csv", sep = ",")

# Eliminating String Features to be able to run Correlation Analysis
# Eliminating all target features except new_
new_hitter_df = df.drop(['player_id', 'new_run', 'new_rbi', 'new_sb', 'new_walk', 'new_k', 'new_tb'], axis=1)

In [3]:
import pandas as pd

# Calculate the correlation matrix
correlation_matrix = new_hitter_df.corr()

# Select the target variable column
target_variable = "new_yahoo" 

# Get correlations with the target variable
target_correlations = correlation_matrix[target_variable].sort_values(ascending=False)

# Print the correlations (optional)
print(target_correlations)

new_yahoo                  1.000000
yahoo                      0.522700
xwoba                      0.449531
xslg                       0.448443
r_run                      0.408151
b_total_bases              0.407216
on_base_plus_slg           0.404336
slg_percent                0.390600
woba                       0.375782
xiso                       0.364830
xobp                       0.335147
isolated_power             0.333676
b_rbi                      0.326192
home_run                   0.325916
z_swing_percent            0.322706
double                     0.320908
pa                         0.318232
xba                        0.315213
r_total_caught_stealing    0.302892
exit_velocity_avg          0.300173
walk                       0.290006
on_base_percent            0.267854
hit                        0.263933
barrel_batted_rate         0.262450
hard_hit_percent           0.259519
ab                         0.255428
flyballs_percent           0.244957
r_total_stolen_base        0

### Model 1 - Single Feature Model

In [13]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(new_hitter_df,
test_size=0.2, random_state=123)
print(len(train_set), len(test_set))

from sklearn.linear_model import LinearRegression
reg = LinearRegression()

X = train_set[['xwoba']]
y = train_set['new_yahoo']
reg.fit(X, y)

print("The bias is " , reg.intercept_)
print("The feature coefficients are ", reg.coef_)
print("The score for the training set is", reg.score(X,y))

# Check the performance on the test set
X_test = test_set[['xwoba']]
y_test = test_set['new_yahoo']
print("The score for the test set is", reg.score(X_test,y_test))

68 18
The bias is  -307.51477752641796
The feature coefficients are  [2021.02628798]
The score for the training set is 0.28859361664288485
The score for the test set is -0.31529686402395574


#### Simple Linear Model Performance
| Feature   | Training   | Test    |
| -----     | -----      | -----   |
| k_percent | 0.12 | 0.15 |
| strikeout | 0.11 | 0.12 |
| xwoba | 0.14 | 0.07 |
| xobp | 0.13 | 0.09 |
| last_year_Yahoo | 0.12 | 0.08 |
| fastball_avg_spin | 0.05 | 0.09 |





### Model 2 - Multi-Feature Model

In [23]:
X = train_set[['k_percent', 'strikeout', 'iz_contact_percent']]
y = train_set['new_yahoo']
reg.fit(X, y)

print("The bias is " , reg.intercept_)
print("The feature coefficients are ", reg.coef_)
print("The score for the training set is", reg.score(X,y))

# Check the performance on the test set
X_test = test_set[['k_percent', 'strikeout', 'iz_contact_percent']]
y_test = test_set['new_yahoo']
print("The score for the test set is", reg.score(X_test,y_test))

The bias is  -49.019435671255735
The feature coefficients are  [ 1.22014418e+01  5.64098593e-01 -7.60211319e-03]
The score for the training set is 0.344415538538694
The score for the test set is 0.37022350440730767


#### Multi-Feature Model Performance
| # Features | Features | Training   | Test    |
| -----     | -----      | -----   | ----- |
| 2 | strikeout, k_percent | 0.13 | 0.16 |
| 2 | k_percent, xwoba | 0.15 | 0.13 |
| 2 | k_percent, xobp | 0.15 | 0.14 |
| 2 | k_percent, last_year_Yahoo | 0.15 | 0.14 |
| 3 | xobp, k_percent, strikeout | 0.15 | 0.15 |







### Model 3 - Decision Tree Model

In [4]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

# Separate features (X) and target variable (y)
X = new_hitter_df.drop(['new_yahoo'], axis =1)
y = new_hitter_df['new_yahoo']

# Split data into training and testing sets (test_size=0.2 for 20% test data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score

# Create the decision tree regression model
model = DecisionTreeRegressor(max_depth=1)

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on testing data
y_pred = model.predict(X_test)

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

Mean Squared Error: 6885.406982406638
R-squared: 0.15884876608365228


#### Decision Tree Results
| Depth | MSE   | R2    |
| ----- | ----- | ----- |
| 5     | 2632 | -.07?? |
| 4     | 1946 | .20 |
| 3     | 2138 | .13 |
| 2     | 1591 | .35 |
| 1     | 1783 | .27 |


In [14]:
# Access feature importances
feature_importances = model.feature_importances_

# Sort features and importances together by importance (descending order)
sorted_features_and_importances = sorted(zip(X.columns, feature_importances), key=lambda x: x[1], reverse=True)

# Print features in order of importance
for feature, importance in sorted_features_and_importances:
  print(f"{feature}: {importance:.4f}")

k_percent: 0.4685
in_zone_swing_miss: 0.1063
isolated_power: 0.0749
breaking_avg_break_x: 0.0500
p_total_bases: 0.0457
p_ball: 0.0451
exit_velocity_avg: 0.0342
z_swing_percent: 0.0331
n_breaking_formatted: 0.0248
p_win: 0.0200
out_zone_swing_miss: 0.0188
offspeed_avg_speed: 0.0144
bb_percent: 0.0137
breaking_range_speed: 0.0135
solidcontact_percent: 0.0126
out_zone: 0.0116
flyballs_percent: 0.0057
iz_contact_percent: 0.0040
breaking_avg_spin: 0.0015
barrel_batted_rate: 0.0011
n_fastball_formatted: 0.0003
meatball_percent: 0.0003
home_run: 0.0001
p_loss: 0.0000
last_year: 0.0000
player_age: 0.0000
last_year_Yahoo: 0.0000
IP: 0.0000
pa: 0.0000
ab: 0.0000
hit: 0.0000
single: 0.0000
double: 0.0000
triple: 0.0000
strikeout: 0.0000
walk: 0.0000
batting_avg: 0.0000
slg_percent: 0.0000
on_base_percent: 0.0000
on_base_plus_slg: 0.0000
babip: 0.0000
p_earned_run: 0.0000
p_run: 0.0000
p_lob: 0.0000
p_called_strike: 0.0000
p_swinging_strike: 0.0000
p_total_ball: 0.0000
p_total_strike: 0.0000
p_tot

### Model 4 - LASSO Model

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

# Create the LASSO model with alpha (regularization parameter)
model = Lasso(alpha=1)  # Adjust alpha as needed

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on testing data
y_pred = model.predict(X_test)

# Evaluate model performance (e.g., using mean squared error)
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

c:\Users\wadej\Documents\Capstone\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.525e+03, tolerance: 6.484e+01
  model = cd_fast.enet_coordinate_descent(


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- ab
- b_rbi
- b_swinging_strike
- b_total_bases
- babip
- ...


#### Lasso Alpha Analysis
| Alpha | MSE   | R2    |
| ----- | ----- | ----- |
| .001  | 1398 | .4281 |
| .01   | 1401 | .4217 |
| .1    | 1372 | .4388 |
| 1     | 1354 | .4462 |
| 10    | 1255 | .4865 |
| 11    | 1254 | .4871 |

In [91]:
# Access coefficients
lasso_coefficients = model.coef_

# Sort features and coefficients together by absolute coefficient value (descending order)
sorted_features_and_coefficients = sorted(zip(X.columns, lasso_coefficients), key=lambda x: abs(x[1]), reverse=True)

# Print features and coefficients in sorted order
for feature_name, coefficient in sorted_features_and_coefficients:
  print(f"{feature_name}: {coefficient:.4f}")

strikeout: 0.9183
player_age: -0.8347
p_lob: 0.2969
double: 0.2483
pa: -0.1203
p_total_strike: -0.1110
p_ball: 0.1061
p_total_ball: -0.1030
p_total_bases: -0.0764
fastball_avg_spin: 0.0652
in_zone: 0.0499
last_year_Yahoo: -0.0483
out_zone_swing: 0.0468
total_pitches: 0.0400
p_swinging_strike: -0.0256
out_zone: -0.0254
p_total_swinging_strike: 0.0122
offspeed_avg_spin: -0.0096
p_called_strike: -0.0096
pitch_count_breaking: 0.0056
pitch_count_fastball: 0.0047
edge: 0.0014
breaking_avg_spin: 0.0013
ab: -0.0013
pitch_count_offspeed: 0.0002
last_year: -0.0000
IP: -0.0000
hit: 0.0000
single: -0.0000
triple: 0.0000
home_run: -0.0000
walk: 0.0000
k_percent: 0.0000
bb_percent: 0.0000
batting_avg: 0.0000
slg_percent: -0.0000
on_base_percent: 0.0000
on_base_plus_slg: -0.0000
isolated_power: -0.0000
babip: 0.0000
p_earned_run: 0.0000
p_run: 0.0000
p_win: 0.0000
p_loss: 0.0000
xba: -0.0000
xslg: -0.0000
woba: -0.0000
xwoba: -0.0000
xobp: 0.0000
xiso: -0.0000
xbadiff: 0.0000
xslgdiff: 0.0000
wobadif

In [101]:
import joblib
joblib.dump(model, "strikeout_lasso_model.joblib")

['strikeout_lasso_model.joblib']

# Final Results

| Method | Specifics   | R2   |
| ----- | ----- | ----- |
| LASSO | Alpha = 11 | 0.4871 |
| Multi-Feature Regression | Features = strikeout, k_percent | 0.45 / 0.35 |
| Simple Linear Regression | Feature = k_percent | 0.42 / 0.36 |
| Decision Tree  | Depth = 2 | .35 |
| Simple Linear Regression | Feature = strikeout | 0.32 / 0.26 |
